# Link Prediction in Knowledge Graphs via Neighborhood Visualization + VLM

**Единый пайплайн: обучение → замер качества → исследования (по убыванию важности).**

Метод: окрестность сущности графа рендерится в изображение (Graphviz) и подаётся в
vision-language модель, которая предсказывает целевую сущность `(h, r, ?)`.
Бенчмарк: **FB15k-237**.

> **Модель по умолчанию — Qwen2.5-VL-3B-Instruct.** Чекпоинтов LLaVA нет, а Qwen2.5-VL
> превосходит LLaVA-1.5 по всем публичным бенчмаркам при вдвое меньшем размере (3B vs 7B),
> поэтому обучаем с нуля именно её. Переключение на LLaVA — одна константа `USE_MODEL`.

### Структура ноутбука
0. Setup и зависимости
1. Конфигурация (модель, батч под GPU, пути)
2. Проверка данных
3. **Обучение** — Stage 1 (comprehension) → Stage 2 (link prediction)
4. **Замер качества (генеративный)** — MRR / Hits@k на полном тесте
5. **Исследования по убыванию важности:**
   - **T1 (критично)** — доходит ли информация с изображения до LM (абляция + пробы B/C)
   - **T2** — контроль «изображение против сериализации графа в текст»
   - **T3** — абляции рендеринга (звезда/топология, k, разрешение, метки рёбер)
   - **T4** — нужен ли Stage 1 (curriculum)
   - **T5** — классические KGE-бейзлайны (pykeen)
   - **T6** — анализ ошибок по типам отношений / степеням
6. Итоговая сводка

### Бюджет времени на A100 (оценка, полный датасет)
| Этап | Оценка |
|---|---|
| Обучение Stage 1 + Stage 2 | **~8–9 ч** |
| Замер качества генеративный (раздел 4) — **весь тест 19 913** | **~8–11 ч** |
| T1 (a+b+c) — вклад изображения | ~1.5–2 ч |
| T2 — текст vs изображение | ~25–40 мин |
| T5 — KGE-бейзлайны | ~40–60 мин |
| T6 — анализ ошибок | <1 мин |
| **Итого «обязательная» часть (обучение + T1/T2/T5/T6)** | **~19–24 ч** |
| T3 — абляции рендеринга (опц.) | ~1.5–2 ч/вариант |
| T4 — Stage 1 on/off (опц.) | ~3 ч |

> Замер качества (раздел 4) идёт на полном тесте — это headline-число статьи (~8–11 ч, один раз).
> Абляции T1a/T2 — на подвыборке 1000 ради времени (относительные разницы значимы).
> Оценки для Qwen2.5-VL-3B при ~720 визуальных токенах. На RTX 3090 умножайте ориентировочно ×3–4.
>
> **Ускорить полный тест** (если ~8–11 ч много): уменьшите `eval_num_beams` (10→5)
> и/или растите `eval_batch_size` — счёт ускорится в ~1.5–2 раза почти без потери MRR.

## 0. Setup и зависимости

Требования (закреплённые стабильные версии, все ≥1 недели):
```bash
pip install "transformers>=4.49,<5.0" "peft==0.19.1" "accelerate==1.14.0" \
            bitsandbytes datasets pillow graphviz "pykeen==1.11.1" matplotlib pandas neo4j
```
- `transformers` берём из ветки 4.x: там уже есть класс `Qwen2_5_VLForConditionalGeneration`,
  и при этом `Trainer(tokenizer=...)` ещё поддерживается (в 5.x он переименован в `processing_class`).
- `bitsandbytes` — для QLoRA (4-bit). `pykeen` — только для KGE-бейзлайнов (T5).

> Ноутбук должен лежать в **корне репозитория**, а папка репозитория называться `graphvis_lp`
> (весь код импортируется как пакет `graphvis_lp.*`).

**Ячейка:** импорт пакета, проверка версий и GPU, инициализация сидов.
**⏱ A100:** ~10 сек (без загрузки весов).

In [ ]:
# ⏱ A100 ~10 сек — только импорты и проверка окружения.
import os, sys, gc, json, time, random, logging, dataclasses
from pathlib import Path
import numpy as np
import torch

REPO = Path.cwd()
sys.path.insert(0, str(REPO.parent))          # чтобы работал import graphvis_lp.*
import graphvis_lp                              # noqa: F401
from graphvis_lp.config import TrainConfig

import transformers, peft
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
print("transformers:", transformers.__version__, "| peft:", peft.__version__)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

RESULTS = {}   # сюда складываем метрики всех экспериментов для итоговой сводки

## 1. Конфигурация

`USE_MODEL="qwen"` — рекомендуемый путь. `QUICK=True` — быстрый smoke-тест на подвыборке
(проверить, что весь пайплайн проходит, за минуты, а не часы).

**Ячейка:** задаёт модель, пути, размер батча под конкретную GPU.
**⏱ A100:** мгновенно.

In [ ]:
# ⏱ A100 мгновенно — только настройка конфигурации.
USE_MODEL = "qwen"      # "qwen" (рекомендуется) | "llava"
QUICK     = False       # True → крошечная подвыборка для проверки пайплайна

cfg = TrainConfig()
cfg.dataset_root = str(REPO / "graphvis_fb15k_data" / "datasets")
cfg.image_root   = ""                       # image_path в датасете абсолютный
cfg.output_dir   = str(REPO / "graphvis_ckpts")

cfg.model_name = ("Qwen/Qwen2.5-VL-3B-Instruct" if USE_MODEL == "qwen"
                  else "llava-hf/llava-1.5-7b-hf")
cfg.model_family = "auto"

# Батч под конкретную карту (QLoRA, 4-bit). При OOM — уменьшайте batch, растите accum.
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
if "A100" in gpu or "H100" in gpu:
    cfg.per_device_train_batch_size = 8 if USE_MODEL == "qwen" else 16
    cfg.gradient_accumulation_steps = 2
elif any(x in gpu for x in ("3090", "4090", "A6000", "A5000")):
    cfg.per_device_train_batch_size = 4 if USE_MODEL == "qwen" else 6
    cfg.gradient_accumulation_steps = 4
else:
    cfg.per_device_train_batch_size = 2
    cfg.gradient_accumulation_steps = 8
cfg.per_device_eval_batch_size = cfg.per_device_train_batch_size
cfg.eval_batch_size = cfg.per_device_train_batch_size

if QUICK:
    cfg.stage1_max_samples = 400
    cfg.stage2_max_samples = 800
    cfg.eval_max_samples   = 100

from graphvis_lp.training.families import detect_family
FAMILY = detect_family(cfg.model_name, cfg.model_family)
print(f"family={FAMILY} | model={cfg.model_name}")
print(f"train batch={cfg.per_device_train_batch_size} x accum={cfg.gradient_accumulation_steps}"
      f" (эфф. {cfg.per_device_train_batch_size*cfg.gradient_accumulation_steps})")
print("output_dir:", cfg.output_dir)

## 2. Проверка данных

Загружаем собранные сплиты и смотрим один пример: промпт, ответ и само изображение окрестности.

**Ячейка:** загрузка Arrow-датасетов с диска + визуальная проверка одного примера.
**⏱ A100:** ~5–15 сек (данные уже собраны, идёт только чтение с диска).

In [ ]:
# ⏱ A100 ~5-15 сек — чтение готовых датасетов с диска.
from datasets import load_from_disk
from PIL import Image

splits = {}
for n in ["stage1_graph_comprehension", "stage2_train", "stage2_valid", "stage2_test"]:
    p = os.path.join(cfg.dataset_root, n)
    if os.path.exists(p):
        key = "stage1" if "stage1" in n else n
        splits[key] = load_from_disk(p)
        print(f"{key:22s}: {len(splits[key]):>7,} примеров")
    else:
        print(f"[нет] {p}")

ex = splits["stage2_test"][0]
print("\n--- пример link_prediction ---")
print("prompt :", ex["prompt"][:220], "...")
print("head   :", ex["head"], "| relation:", ex["relation"], "| answer:", ex["answer"])
Image.open(ex["image_path"]).convert("RGB")

## 3. Обучение (curriculum: Stage 1 → Stage 2)

- **Stage 1 — graph comprehension**: модель учится «читать» картинку графа (сколько узлов,
  какие рёбра, степени). Формирует навык восприятия визуальной структуры.
- **Stage 2 — link prediction**: та же модель дообучается предсказывать `(h, r, ?)`.

QLoRA (4-bit NF4 + LoRA-адаптеры). Обучение идёт на одном объекте `model` — Stage 2
продолжает уже адаптированную Stage 1 модель. Адаптеры сохраняются в `graphvis_ckpts/`.

**Ячейка (загрузка модели):** скачивает веса Qwen2.5-VL-3B (~7 ГБ, один раз) и инициализирует QLoRA.
**⏱ A100:** ~3–5 мин при первом запуске (скачивание+загрузка), ~1 мин при повторном (из кэша).

In [ ]:
# ⏱ A100 ~3-5 мин (первое скачивание весов ~7 ГБ), затем ~1 мин из кэша.
from graphvis_lp.training.model import build_model_and_processor
from graphvis_lp.training.dataset import GraphVisLVLMDataset
from graphvis_lp.training.trainer import make_trainer

model, processor, tokenizer = build_model_and_processor(cfg)
print("Модель загружена.")

**Ячейка (Stage 1 — comprehension):** обучение «читать» граф на изображении
(счёт узлов/рёбер, степени) — 155 872 примера, 1 эпоха.
**⏱ A100:** ~5–6 ч на полном датасете (Qwen-3B, ~720 визуальных токенов). `QUICK=True` → ~2 мин.

In [ ]:
# ⏱ A100 ~5-6 ч (155 872 примера, 1 эпоха; QUICK → ~2 мин).
# ── Stage 1 ──
RUN_STAGE1 = True
if RUN_STAGE1 and "stage1" in splits:
    s1 = splits["stage1"]
    if cfg.stage1_max_samples and len(s1) > cfg.stage1_max_samples:
        s1 = s1.select(random.sample(range(len(s1)), cfg.stage1_max_samples))
    s1_ds = GraphVisLVLMDataset(s1, processor, tokenizer, cfg)
    tr1 = make_trainer(model, s1_ds, None, cfg, "stage1", tokenizer)
    t0 = time.time()
    tr1.train()
    tr1.save_model()
    print(f"Stage 1 готов за {(time.time()-t0)/60:.1f} мин -> {cfg.output_dir}/stage1")

**Ячейка (Stage 2 — link prediction):** дообучение предсказывать `(h, r, ?)` —
39 005 примеров, 2 эпохи, с валидацией и early-stopping.
**⏱ A100:** ~2.5–3 ч на полном датасете. `QUICK=True` → ~3 мин.

In [ ]:
# ⏱ A100 ~2.5-3 ч (39 005 примеров × 2 эпохи; QUICK → ~3 мин).
# ── Stage 2 ──
s2 = splits["stage2_train"]
if cfg.stage2_max_samples and len(s2) > cfg.stage2_max_samples:
    s2 = s2.select(random.sample(range(len(s2)), cfg.stage2_max_samples))
s2_ds  = GraphVisLVLMDataset(s2, processor, tokenizer, cfg)
s2_val = (GraphVisLVLMDataset(splits["stage2_valid"], processor, tokenizer, cfg)
          if "stage2_valid" in splits else None)

tr2 = make_trainer(model, s2_ds, s2_val, cfg, "stage2", tokenizer)
t0 = time.time()
tr2.train()
tr2.save_model()
CKPT = os.path.join(cfg.output_dir, "stage2")
print(f"Stage 2 готов за {(time.time()-t0)/60:.1f} мин -> {CKPT}")

## 4. Замер качества (headline-метрики)

Генеративное ранжирование beam-search'ем на тесте: **MRR, Hits@1/3/10**.

> **Замечание для статьи.** Это *генеративный* протокол (модель порождает имя сущности).
> Стандарт KGC — *filtered ranking* по всем кандидатам. В разделе T5 приведены KGE под
> filtered-протоколом; при подготовке статьи стоит либо реализовать filtered-ранжирование
> для VLM, либо явно обосновать генеративный протокол и его сопоставимость.

**Ячейка:** headline-метрика на **ПОЛНОМ тесте** (все 19 913 запросов) + 95% доверительный
интервал MRR. Это число идёт в главную таблицу статьи.
**⏱ A100:** ~8–11 ч (beam=10, ~0.5–0.7 запроса/с). Считается один раз.

In [ ]:
# ⏱ A100 ~8-11 ч — ПОЛНЫЙ тест (19 913 запросов, beam=10). Headline-метрика статьи.
from graphvis_lp.training.evaluate import evaluate_link_prediction

cfg.eval_max_samples = len(splits["stage2_test"])   # ВЕСЬ тест, без подвыборки
metrics_real = evaluate_link_prediction(
    model, processor, tokenizer, splits["stage2_test"], cfg, image_mode="real")
RESULTS["main_real"] = metrics_real

# 95% доверительный интервал MRR (по сохранённым рангам)
det0 = json.load(open(os.path.join(cfg.output_dir, "test_details.json")))
rr = np.array([1.0 / d["rank"] if d["rank"] and d["rank"] > 0 else 0.0 for d in det0])
ci = 1.96 * rr.std(ddof=1) / np.sqrt(len(rr))
RESULTS["main_real"]["MRR_ci95"] = float(ci)
print(f"MRR = {metrics_real['MRR']:.4f} ± {ci:.4f}  (95% ДИ, n={len(rr):,})")
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in metrics_real.items() if k in ("MRR","Hits@1","Hits@3","Hits@10","n_evaluated")})

## 5. Исследования (по убыванию важности)

---
### T1 — КРИТИЧНО: доходит ли информация с изображения до LM?

Главный вопрос рецензента: «а картинка вообще нужна, или модель рассуждает по тексту промпта?».
Три независимых свидетельства:

- **T1a — поведенческая абляция** (real / blank / noise / shuffle). `blank` ≈ «текст-онли»
  (модель видит только вопрос `h, r, ?`, структура недоступна). Падение MRR = вклад картинки.
- **T1b — причинная чувствительность выхода** (teacher-forcing): `ΔNLL` и симметричная `KL`
  между предсказаниями при реальном и абляционном изображении. Не зависит от beam-search.
- **T1c — доля внимания на визуальные токены** (нужен `attn_implementation="eager"`).

**Ячейка (T1a):** сравнивает 4 режима картинки (real/blank/noise/shuffle) на **одной и той же
подвыборке 1000** (все режимы пересчитаны честно на одном наборе) и считает падение MRR
относительно `real` — прямой поведенческий вклад изображения.
**⏱ A100:** ~1.5–2 ч (4 прогона по ~20–30 мин). Абляции на подвыборке — ради времени;
относительные разницы значимы (SE MRR ≲ 0.015).

In [ ]:
# ⏱ A100 ~1.5-2 ч — 4 оценки (real/blank/noise/shuffle) на общей подвыборке 1000.
# T1a — поведенческая абляция модальности (подвыборка ради времени)
cfg.eval_max_samples = 1000            # абляции — на 1000, не на полном тесте
abl = {}
for mode in ["real", "blank", "noise", "shuffle"]:
    abl[mode] = evaluate_link_prediction(
        model, processor, tokenizer, splits["stage2_test"], cfg, image_mode=mode)

import pandas as pd
rows = []
base = abl["real"]["MRR"]
for m in ["real", "blank", "noise", "shuffle"]:
    r = abl[m]
    rows.append({"mode": m, "MRR": r["MRR"], "Hits@1": r["Hits@1"],
                 "Hits@10": r["Hits@10"], "dMRR_vs_real": base - r["MRR"]})
df_t1a = pd.DataFrame(rows).set_index("mode").round(4)
RESULTS["T1a_ablation"] = df_t1a.to_dict()
print("real ≫ blank/shuffle  ⇒  информация с картинки доходит до LM")
df_t1a

**Ячейка (T1b):** teacher-forcing на 100 запросах × 4 режима — считает `ΔNLL` (насколько
картинка снижает ошибку на золотом ответе) и симметричную `KL` (насколько меняет распределение).
Без beam-search, поэтому быстро.
**⏱ A100:** ~3–5 мин (100 примеров × 4 режима, только forward-проходы).

In [ ]:
# ⏱ A100 ~3-5 мин — forward-проходы на 100 примерах × 4 режима (без генерации).
# T1b — причинная чувствительность (ΔNLL, symKL). Работает на текущей модели.
from graphvis_lp.scripts.image_influence import probe_causal, probe_attention

test = splits["stage2_test"]
N_PROBE = min(100 if not QUICK else 20, len(test))
idx = sorted(np.random.RandomState(42).choice(len(test), size=N_PROBE, replace=False).tolist())
device = next(model.parameters()).device

resB = probe_causal(model, processor, tokenizer, cfg, test, idx, FAMILY, device)
RESULTS["T1b_causal"] = resB
pd.DataFrame(resB).T

**Ячейка (T1c):** перезагружает модель с `attn_implementation="eager"` и на тех же 100 запросах
измеряет долю внимания ответных позиций на визуальные токены по слоям (сравнение со случайной долей).
**⏱ A100:** ~8–12 мин (повторная загрузка весов ~2–3 мин + прогон с output_attentions).

In [ ]:
# ⏱ A100 ~8-12 мин — перезагрузка модели (eager) + attention на 100 примерах.
# T1c — доля внимания на изображение. Требует eager-attention → отдельная загрузка из CKPT.
from peft import PeftModel
gc.collect(); torch.cuda.empty_cache()

cfg_eager = dataclasses.replace(cfg, attn_implementation="eager", use_lora=False)
m_eager, proc_e, tok_e = build_model_and_processor(cfg_eager)
m_eager = PeftModel.from_pretrained(m_eager, CKPT).eval()

resC = probe_attention(m_eager, proc_e, tok_e, cfg_eager, test, idx, FAMILY,
                       next(m_eager.parameters()).device)
RESULTS["T1c_attention"] = resC
del m_eager; gc.collect(); torch.cuda.empty_cache()
resC

**Как читать T1:** если `real`-MRR заметно выше `blank`/`shuffle` (T1a), `ΔNLL>0` и `symKL>0`
(T1b), а доля внимания на изображение выше случайной, т.е. `ratio_over_chance>1` (T1c) —
у вас есть тройное доказательство, что визуальная модальность реально используется. Это ядро статьи.

---
### T2 — Изображение против сериализации графа в текст

Ключевой контроль новизны: возможно, ту же информацию проще подать текстом (списком рёбер)?
Сравниваем на **одной и той же подвыборке** три условия:
- `real` — визуализация (наш метод);
- `text-serialized` — та же окрестность списком рёбер в промпте + пустое изображение;
- `blank` — только вопрос без структуры (нижняя граница).

> Требуется запущенный **Neo4j** (тот же граф, что при сборке датасета). При необходимости
> задайте `NEO4J_URI/USER/PASSWORD` в окружении.

**Ячейка (T2):** на подвыборке (~400) сериализует окрестность в список рёбер через Neo4j,
затем сравнивает три условия: `real` (картинка), `text-serialized` (рёбра текстом + пустая картинка),
`blank` (только вопрос). Показывает, даёт ли визуализация выигрыш над текстовой подачей.
**⏱ A100:** ~25–40 мин (запросы к Neo4j + три beam-оценки на 400 примерах).

In [ ]:
# ⏱ A100 ~25-40 мин — сериализация через Neo4j + 3 beam-оценки на ~400 примерах.
RUN_T2 = True   # требует Neo4j
if RUN_T2:
    from datasets import Dataset
    from graphvis_lp.config import RetrieverConfig, Neo4jConfig
    from graphvis_lp.retriever import FB15k237GraphRetriever

    SUB_N = min(400 if not QUICK else 50, len(test))
    sub_idx = sorted(np.random.RandomState(7).choice(len(test), size=SUB_N, replace=False).tolist())
    sub = test.select(sub_idx)

    def serialize_edges(edges, max_edges=25):
        lines = [f"{e['source']} --{e['relation']}--> {e['target']}" for e in edges[:max_edges]]
        return "\n".join(lines)

    neo = Neo4jConfig.from_env()
    rcfg = RetrieverConfig(neo4j=neo)
    prompts_txt = []
    with FB15k237GraphRetriever(rcfg) as retr:
        for row in sub:
            sg = retr.get_head_subgraph(row["head"], row["relation"], row["tail"], split="test")
            edges = serialize_edges(sg.get("edges", []))
            q = (f"Below is the neighborhood of entity '{row['head']}' as an edge list.\n"
                 f"{edges}\n\n"
                 f"Predict the entity that '{row['head']}' connects to via relation "
                 f"'{row['relation']}'. Answer with the entity name only.")
            prompts_txt.append(q)

    sub_txt = Dataset.from_dict({
        "image_path": sub["image_path"], "prompt": prompts_txt,
        "answer": sub["answer"], "head": sub["head"],
        "relation": sub["relation"], "tail": sub["tail"],
    })

    cfg_sub = dataclasses.replace(cfg, eval_max_samples=SUB_N)
    r_real = evaluate_link_prediction(model, processor, tokenizer, sub,     cfg_sub, image_mode="real")
    r_txt  = evaluate_link_prediction(model, processor, tokenizer, sub_txt, cfg_sub, image_mode="blank")
    r_blank= evaluate_link_prediction(model, processor, tokenizer, sub,     cfg_sub, image_mode="blank")

    df_t2 = pd.DataFrame({
        "image (наш метод)":  {k: r_real[k]  for k in ("MRR","Hits@1","Hits@10")},
        "text-serialized":    {k: r_txt[k]   for k in ("MRR","Hits@1","Hits@10")},
        "blank (только вопрос)":{k: r_blank[k]for k in ("MRR","Hits@1","Hits@10")},
    }).T.round(4)
    RESULTS["T2_text_vs_image"] = df_t2.to_dict()
    df_t2

---
### T3 — Абляции рендеринга (дорого: пересборка датасета + дообучение)

Что именно в визуализации важно. Каждый вариант = пересобрать небольшой датасет с изменённым
`RetrieverConfig`/`VisualizationConfig`, дообучить адаптер, замерить MRR:

| Вариант | Что меняем |
|---|---|
| star vs topology | `include_neighbor_edges` (звезда против истинной топологии) |
| k_neighbors | 4 / 8 / 12 / 20 (размер окрестности) |
| resolution | `target_size` 336 / 448 / 672 |
| edge labels | рисовать ли имена отношений |

Ниже — параметризованный каркас. Требует Neo4j и времени; включайте по одному варианту.

**Ячейка (T3):** для каждого варианта — пересборка небольшого датасета (Neo4j + рендер),
дообучение свежего адаптера (без Stage 1, 2 эпохи) и замер MRR.
**⏱ A100:** ~1.5–2 ч **на один вариант** (сборка ~15–20 мин + обучение ~1–1.5 ч + оценка ~10 мин);
все 5 вариантов ≈ рабочий день. Держите `RUN_T3=False` и включайте точечно.

In [ ]:
# ⏱ A100 ~1.5-2 ч НА ВАРИАНТ (5 вариантов ≈ целый день). По умолчанию выключено.
RUN_T3 = False   # ДОРОГО: пересборка + дообучение на каждый вариант
if RUN_T3:
    from graphvis_lp.builder import FB15k237GraphVisBuilder
    from graphvis_lp.config import (BuilderConfig, Neo4jConfig, RetrieverConfig,
                                     VisualizationConfig)

    def run_variant(name, retr_over=None, vis_over=None, n_train=4000, n_test=1500):
        neo = Neo4jConfig.from_env()
        rcfg = RetrieverConfig(neo4j=neo, **(retr_over or {}))
        vout = str(REPO / "graphvis_variants" / name)
        vcfg = VisualizationConfig(output_dir=vout, **(vis_over or {}))
        bcfg = BuilderConfig(neo4j=neo, retriever=rcfg, visualization=vcfg,
                             num_visualization_workers=12)
        dsdir = Path(vout) / "datasets"
        # 1) сборка небольшого train/test
        with FB15k237GraphVisBuilder(bcfg) as b:
            b._load_entity_names()
            tr = b._load_triples_from_file(os.path.join(bcfg.data_dir, bcfg.train_file), n_train)
            te = b._load_triples_from_file(os.path.join(bcfg.data_dir, bcfg.test_file),  n_test)
            b.build_stage2(tr, "train", str(dsdir / "stage2_train"))
            b.build_stage2(te, "test",  str(dsdir / "stage2_test"))
        # 2) дообучение с нуля (без Stage 1 — для скорости сравнения вариантов)
        vcfg2 = dataclasses.replace(cfg, dataset_root=str(dsdir),
                                    stage1_max_samples=0, stage2_num_epochs=2)
        m, p, t = build_model_and_processor(vcfg2)
        tr_ds = GraphVisLVLMDataset(load_from_disk(str(dsdir/"stage2_train")), p, t, vcfg2)
        trn = make_trainer(m, tr_ds, None, vcfg2, f"var_{name}", t); trn.train()
        met = evaluate_link_prediction(m, p, t, load_from_disk(str(dsdir/"stage2_test")), vcfg2)
        del m, trn; gc.collect(); torch.cuda.empty_cache()
        return met

    variants = {
        "topology_on":  dict(retr_over={"include_neighbor_edges": True}),
        "topology_off": dict(retr_over={"include_neighbor_edges": False}),
        "k4":           dict(retr_over={"k_neighbors": 4}),
        "k20":          dict(retr_over={"k_neighbors": 20}),
        "res448":       dict(vis_over={"target_size": 448}),
    }
    t3 = {}
    for nm, ov in variants.items():
        print("=== variant", nm, "===")
        t3[nm] = run_variant(nm, **ov)["MRR"]
    RESULTS["T3_rendering"] = t3
    pd.Series(t3, name="MRR").sort_values(ascending=False)

---
### T4 — Нужен ли Stage 1 (curriculum)?

Дообучаем Stage 2 **с нуля**, без comprehension-претрейна, и сравниваем с основным
результатом (раздел 4). Разница = вклад curriculum.

**Ячейка (T4):** свежая модель → только Stage 2 → оценка; сравнение MRR с/без Stage 1.
**⏱ A100:** ~3 ч (полный Stage 2 ~2.5–3 ч + оценка ~20 мин). По умолчанию выключено.

In [ ]:
# ⏱ A100 ~3 ч — полный Stage 2 без curriculum + оценка. По умолчанию выключено.
RUN_T4 = False   # ещё один полный прогон Stage 2
if RUN_T4:
    m0, p0, t0_ = build_model_and_processor(cfg)     # свежая модель, без Stage 1
    s2_only = GraphVisLVLMDataset(s2, p0, t0_, cfg)
    tr_only = make_trainer(m0, s2_only, s2_val, cfg, "stage2_nocurriculum", t0_)
    tr_only.train()
    met_nc = evaluate_link_prediction(m0, p0, t0_, splits["stage2_test"], cfg)
    RESULTS["T4_no_stage1"] = {"MRR_no_stage1": met_nc["MRR"],
                               "MRR_with_stage1": metrics_real["MRR"],
                               "delta": metrics_real["MRR"] - met_nc["MRR"]}
    del m0, tr_only; gc.collect(); torch.cuda.empty_cache()
    RESULTS["T4_no_stage1"]

---
### T5 — Классические KGE-бейзлайны (pykeen)

Референс-числа поля под стандартным **filtered**-протоколом: TransE / DistMult / ComplEx / RotatE
на FB15k-237. Независимо от VLM (быстро на одной GPU).

> (full-entity, фильтрация известных троек) → сравнивать VLM с KGE и с числами из вашего текста

**Ячейка (T5):** обучает 4 KGE-модели через pykeen (50 эпох) и печатает filtered MRR/Hits@k.
**⏱ A100:** ~40–60 мин суммарно (~10–15 мин на модель). `QUICK=True` → ~5 мин.

In [ ]:
# ⏱ A100 ~40-60 мин — 4 KGE-модели × 50 эпох (QUICK → ~5 мин).
RUN_T5 = True
if RUN_T5:
    from pykeen.pipeline import pipeline
    kge = {}
    for mdl in ["TransE", "DistMult", "ComplEx", "RotatE"]:
        res = pipeline(dataset="FB15k237", model=mdl,
                       training_kwargs=dict(num_epochs=50 if not QUICK else 3),
                       random_seed=42, device="cuda" if torch.cuda.is_available() else "cpu")
        m = res.metric_results.to_dict()
        kge[mdl] = {
            "MRR":     res.get_metric("inverse_harmonic_mean_rank"),
            "Hits@1":  res.get_metric("hits_at_1"),
            "Hits@3":  res.get_metric("hits_at_3"),
            "Hits@10": res.get_metric("hits_at_10"),
        }
        print(mdl, {k: round(v,4) for k,v in kge[mdl].items()})
    RESULTS["T5_kge"] = kge
    pd.DataFrame(kge).T.round(4)

---
### T6 — Анализ ошибок

По сохранённому `test_details.json` (из раздела 4) считаем точность (found@rank1) в разрезе
типов отношений и частоты ответов — где метод силён, где ломается.

**Ячейка (T6):** читает сохранённый `test_details.json`, агрегирует Hits@1 по отношениям
и по частоте ответа. Чистый анализ на CPU.
**⏱ A100:** <1 мин.

In [ ]:
# ⏱ A100 <1 мин — агрегация уже сохранённых результатов (CPU).
det_path = os.path.join(cfg.output_dir, "test_details.json")
if os.path.exists(det_path):
    import pandas as pd
    det = pd.DataFrame(json.load(open(det_path)))
    tset = splits["stage2_test"]
    meta = pd.DataFrame({"idx": range(len(tset)),
                         "relation": tset["relation"], "answer": tset["answer"]})
    det = det.merge(meta, on="idx", how="left")
    det["hit1"] = det["rank"] == 1

    by_rel = (det.groupby("relation")["hit1"].agg(["mean","count"])
              .sort_values("count", ascending=False).head(15).round(3))
    print("Точность Hits@1 по самым частым отношениям:")
    display(by_rel)

    det["ans_freq"] = det.groupby("answer")["answer"].transform("count")
    det["freq_bin"] = pd.cut(det["ans_freq"], [0,1,3,10,1e9],
                             labels=["уник.","редкие","средние","частые"])
    print("\nТочность по частоте ответа (популярные сущности угадывать легче):")
    display(det.groupby("freq_bin")["hit1"].agg(["mean","count"]).round(3))
    RESULTS["T6_by_relation"] = by_rel.to_dict()
else:
    print("Нет test_details.json — сначала выполните раздел 4.")

## 6. Итоговая сводка

Собираем ключевые метрики всех экспериментов и сохраняем в `graphvis_ckpts/research_results.json`.

**Ячейка:** сводная таблица + барчарт MRR по всем условиям и бейзлайнам, запись результатов на диск.
**⏱ A100:** <1 мин.

In [ ]:
# ⏱ A100 <1 мин — сборка сводки и график.
import pandas as pd, matplotlib.pyplot as plt

summary = {"model": cfg.model_name, "family": FAMILY}
summary["MRR_generative(full test)"] = RESULTS.get("main_real", {}).get("MRR")
if "filtered_all" in RESULTS:
    summary["MRR_filtered_full-entity"] = RESULTS["filtered_all"]["MRR"]
    summary["MRR_filtered_ci95"]        = RESULTS["filtered_all"]["MRR_ci95"]
if "filtered_reltails_full" in RESULTS:
    summary["MRR_filtered_type-constrained"] = RESULTS["filtered_reltails_full"]["MRR"]
if "T1a_ablation" in RESULTS:
    summary["MRR_blank"]   = RESULTS["T1a_ablation"]["MRR"].get("blank")
    summary["MRR_shuffle"] = RESULTS["T1a_ablation"]["MRR"].get("shuffle")
if "T2_text_vs_image" in RESULTS:
    summary["MRR_text_serialized"] = RESULTS["T2_text_vs_image"]["MRR"].get("text-serialized")
if "T5_kge" in RESULTS:
    for k, v in RESULTS["T5_kge"].items():
        summary[f"MRR_{k}"] = v["MRR"]

print(json.dumps({k: (round(v,4) if isinstance(v,float) else v)
                  for k,v in summary.items()}, indent=2, ensure_ascii=False))

out = os.path.join(cfg.output_dir, "research_results.json")
json.dump(RESULTS, open(out, "w"), indent=2, ensure_ascii=False, default=str)
print("\nСохранено:", out)

# Барчарт MRR по доступным условиям
bars = {k.replace("MRR_",""): v for k,v in summary.items()
        if k.startswith("MRR") and "ci95" not in k and isinstance(v,(int,float)) and v is not None}
if bars:
    plt.figure(figsize=(9,4))
    plt.bar(range(len(bars)), list(bars.values()))
    plt.xticks(range(len(bars)), list(bars.keys()), rotation=40, ha="right")
    plt.ylabel("MRR"); plt.title("MRR: наш метод vs абляции vs бейзлайны")
    plt.tight_layout(); plt.show()

---
### Что делать с результатами (чек-лист под статью)

1. **T1 положителен** (картинка используется) → это ядро. Оформляете абляцию + пробы как главный вклад.
2. **T2**: `image > text-serialized` → обоснование новизны (визуальный интерфейс лучше текстовой сериализации).
   Если `image ≈ text` — разворот в анализ «когда визуализация помогает».
3. **T5**: помещаете VLM-числа рядом с KGE — контекст для рецензента.
4. **T3/T4**: рецепт (какой рендеринг и нужен ли curriculum) — раздел «ablation study».
5. **T6**: качественный анализ + примеры attention-хитмэпов.
6. Для полноценной статьи добавьте **второй датасет** (WN18RR/CoDEx-S) и, как отличительную фишку,
   **индуктивный сценарий** (unseen entities) — визуализация окрестности естественно его поддерживает.